# 國立陽明交通大學 - 多模態影像資料處理 (MMIP) 2026 Fall
## Assignment 08：檢索增強生成（RAG）— 多輪對話 RAG、VideoRAG、RAGAS 評估、GraphRAG

- **學生**：林孟霆（`315833033`）
- **作業規範**：第十堂課程簡報第 72–75 頁
- **完整文字報告與結果分析**：請見同目錄 `README.md`

| 部分 | 內容 | 模型 |
| :--- | :--- | :--- |
| 作業一 | 台灣勞動法規（14 部法規、842 條）多輪對話 RAG：QA Dataset、Chunk Size / Overlap / Top-K、Prompting Technique、UI | bge-m3、bge-reranker-v2-m3、Qwen3-4B-Instruct-2507 |
| 作業二 | 自行錄製影片的多輪對話 VideoRAG（改編 VideoRAG：K-means 代表畫面 + 子集挑選） | Whisper large-v3-turbo、Chinese-CLIP、Qwen3-VL-2B |
| 作業三 | 以 RAGAS 指標評估作業一；延伸到 VideoRAG（影片檢索、時間定位、多模態忠實度） | Gemini 評審（沒有 Key 時用 Qwen3-4B） |
| 作業四 | 把 Self-RAG 論文 Graph 化，25 題 Benchmark 比較 GraphRAG 與一般 RAG | Qwen3-4B、bge-m3、networkx |

### 在 Google Colab 執行（T4 GPU，約 2.5–3 小時；每個步驟都有快取，斷線重跑會從中斷處接續）
1. 作業二的影片已經剪好放在 `Data/videos/`（5 支短片：2 支論文報告有說話、3 支專題 demo 操作沒有說話，來源見 README 2-1），評估用的 18 題問答與 2 段多輪對話在 `Data/video_qa.json`。
2. 把 `HW08/` 放到 Google Drive 的 `MyDrive/MMIP/HW08`（原始未剪輯的錄影 `videos/` 不需要上傳）。
3. 「執行階段 → 變更執行階段類型」選 **T4 GPU**；左側「Secrets」加入 `GEMINI_API_KEY`（作業三、四的評審；沒有 Key 改用本地模型評審）。
4. 依序執行。最後兩格是聊天介面（會一直執行，看完手動停止），倒數第三格下載 `HW08_results.zip`。

In [ ]:
%matplotlib inline
import os, sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/MMIP/HW08')
    !pip -q install -r requirements.txt
sys.path.insert(0, os.getcwd())

import torch, pandas as pd, numpy as np
from IPython.display import display, Markdown
from Code.common import gpu_info, out_path, save_json, load_json, free_gpu
from Code.gemini import GeminiJudge, get_api_key
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 220)
print('Working dir:', os.getcwd()); print(gpu_info())
print('GEMINI_API_KEY:', 'found' if get_api_key() else 'not found -> judge = local Qwen3-4B')
gemini = GeminiJudge(out_path('gemini_cache.json'))

---
## 作業一：多輪對話 RAG
### 資料：台灣勞動法規
14 部勞動相關法規（勞動基準法、勞工請假規則、性別平等工作法、勞工退休金條例、就業保險法…），共 842 條，從全國法規資料庫下載（`Code/prepare_data.py`；法規不受著作權保護）。評估資料是手寫的 47 題單輪問答（每題另有一個口語問法）與 11 段 3 輪的多輪對話（`Code/qa_data.py`），每題都標了正確條文。

In [ ]:
from Code import corpus, hw1, rag as R, ragas_eval
from Code.qa_data import SINGLE, DIALOGUES
arts, meta = corpus.load_articles()
display(pd.DataFrame(meta['laws']).T)
print(f"{len(arts)} articles, {sum(len(a['text']) for a in arts):,} characters; evaluation: {len(SINGLE)} questions x 2 wordings + {sum(len(d['turns']) for d in DIALOGUES)} dialogue turns")
print(SINGLE[11]); print(DIALOGUES[1]['turns'][1])

In [ ]:
from Code.embed import Embedder, Reranker
from Code.llm import LLM
emb = Embedder(cache_dir=out_path('emb_cache'))
reranker = Reranker()
llm = LLM(out_path('llm_cache.json'))          # Qwen3-4B is loaded on the first uncached generation

### 作業一進階：找出表現最穩定的 Chunk Size、Overlap Size、Top-K
13 種切塊（固定長度 128 / 256 / 512 / 1024 字 × 重疊 0 / 12.5% / 25%，以及「一條一塊」）× Top-K 1 / 3 / 5 / 10，用 47 題的**正式問法與口語問法**檢索（只用 embedding，不經過 LLM）。
- **Recall@k**：前 k 個 chunk 涵蓋的正確條文比例；**MRR**：第一個正確 chunk 名次的倒數（投影片第 10 頁的計分方式）。
- **穩定度**：(1) 1,000 次 bootstrap 重抽題目後 Recall 的 2.5 百分位數（運氣差時的表現）；(2) 正式與口語問法的差距。
- 選擇規則：檢索內容不超過 3,000 字（4B 模型的輸入預算）的設定中，取 bootstrap 下界最高者，平手再看問法差距與內容長度。

In [ ]:
G, per_query = hw1.grid_search(emb, arts)
best = hw1.pick_best(G)
display(G[G['k'] == 5].sort_values('recall_low95', ascending=False).round(3).head(8))
print('chosen:', best['config'], 'top-k =', int(best['k']), '| recall', round(best['recall'], 3), '| bootstrap low', round(best['recall_low95'], 3))
hw1.plot_grid(G, best)
BEST_CFG = next(c for c in hw1.GRID if corpus.cfg_name(c) == best['config'])
TOP_K = int(best['k'])

### 作業一基礎：自製 QA Dataset 與 Embedding
依投影片第 12 頁「將文本轉換成 QA 形式，並把 Q 轉換成 Embedding」：用 Qwen3-4B 替選定切塊的每個 chunk 產生 3 個 5W1H 角度的問題（含口語問法），與 chunk 本身一起建索引——chunk 的分數取「chunk 本身」與「它的問題」相似度的最大值。QA Dataset = 自動產生的問答 + 47 題手寫問答 + 11 段多輪對話（`Output/hw1_qa_dataset.json`）。

In [ ]:
index, qa = hw1.build_index(emb, llm, arts, BEST_CFG)
counts = hw1.save_dataset(index, qa)
display(pd.Series(counts, name='count'))
print('Embedding 數量 =', index.n_embeddings, '(chunks + synthetic questions)')
for x in qa[:6]: print(f"  Q: {x['question']}  ->  {', '.join(x['articles'])}")

### 檢索強化：合成問題、Re-ranker（兩階段檢索，投影片第 23–30 頁）、Query 轉換（第 16 頁）

In [ ]:
A = hw1.ablation(index, emb, reranker)
display(A.round(3))
# the system uses the retrieval setting with the best recall@TOP_K on the ablation (ties -> MRR)
best_r = A.sort_values([f'recall@{TOP_K}', 'mrr'], ascending=False).iloc[0]['retrieval']
USE_Q, USE_RR = best_r in ('+ synthetic questions', '+ both'), best_r in ('+ re-ranker', '+ both')
print(f'retrieval used by the system: {best_r} (synthetic questions={USE_Q}, re-ranker={USE_RR})')
rag = R.RAG(llm, emb, index, reranker if USE_RR else None, top_k=TOP_K, prompt='grounded', query_mode='rewrite',
            use_questions=USE_Q)
T = hw1.query_transform_eval(rag)
display(T.round(3))

### 作業一基礎：多輪對話經過 RAG 系統後的回覆與 Reference
11 段對話依序送進系統，第 t 輪的歷史是系統自己前幾輪的回答；第 2、3 輪多是「那休息日呢？」「這幾天有薪水嗎？」這類需要上下文的問題，系統先改寫成可獨立檢索的問題，再檢索、回答並標出引用條文（★）。

In [ ]:
dialog_res = hw1.run_dialogues(rag)
for did in ('M02', 'M03', 'M09'):
    print('=' * 100); hw1.show_dialogue(dialog_res, did)
turns = [r for v in dialog_res.values() for r in v]
print(f"\n33 turns: retrieved a gold article {np.mean([r['retrieved_gold'] for r in turns]):.0%}, cited a gold article {np.mean([r['cited_gold'] for r in turns]):.0%}")

### 作業一進階：找出最合適的 Prompting Technique
同樣的檢索結果，比較 4 種 Prompt：`basic`（直接給資料）、`grounded`（角色 + 只能依據資料 + [n] 引用 + 資料不足就說不知道）、`cot`（grounded + 先分析再回答）、`fewshot`（grounded + 一個示範問答），以 RAGAS 指標（定義見作業三）比較，另列不經檢索的 Qwen3-4B 作為對照。

In [ ]:
answers = hw1.answer_single(rag)
judge = ragas_eval.Judge(gemini, llm)
print('judge:', judge.name)
ragas_dfs = {}
for name in ('basic', 'grounded', 'cot', 'fewshot', 'no_rag'):
    ragas_dfs[name], _ = ragas_eval.evaluate(name, answers[name], judge, llm, emb)
S = ragas_eval.summarize(list(ragas_dfs.values()))
display(S.round(3))
cites = {p: np.mean([any(x['cited'] and set(x['articles']) & set(r['gold']) for x in r['references']) for r in answers[p]]) for p in ('basic', 'grounded', 'cot', 'fewshot')}
print('answers that cite a gold article:', {k: f'{v:.0%}' for k, v in cites.items()})

---
## 作業二：多輪對話 VideoRAG
改編自 VideoRAG（投影片第 56–63 頁）：
1. **建立索引**：Whisper 逐字稿（含時間）＋ 每秒一張畫面 → Chinese-CLIP → K-means 選出 64 張代表畫面 → Qwen3-VL-2B 為每張代表畫面寫一句描述（讓沒有說話的操作影片也能用文字檢索）。
2. **影片檢索**：文字片段（bge-m3）與畫面（CLIP）兩種排名做 reciprocal rank fusion。
3. **Frame Selection**：在選中的影片中隨機抽 40 組 6 張的子集，以「子集平均 CLIP 向量 · 問題向量」加上時間先驗評分（論文用訓練好的 MLP，這裡沒有訓練資料，改用 zero-shot 分數）。
4. **生成**：Qwen3-VL-2B 看選出的畫面與文字片段回答，並標出【影片 mm:ss】。多輪對話先把問題改寫成獨立問題。

先釋放 Qwen3-4B 的 GPU 記憶體（作業三、四會自動重新載入）。

In [ ]:
from Code import video as V
llm.unload(); free_gpu()
videos = V.list_videos()
print(len(videos), 'videos:', [os.path.basename(v) for v in videos])
HAS_VIDEO = len(videos) > 0
if not HAS_VIDEO:
    print('Data/videos/ is empty -> record the videos first (Data/videos/README.md); 作業二 and 作業三 advanced are skipped')

In [ ]:
# 作業二：建立影片索引（每支影片的結果快取在 Output/video_index/）
if HAS_VIDEO:
    recs, clip = V.build_index(videos)
    for r in recs:
        print('=' * 100)
        print(f"{r['name']}: {r['duration']:.0f} s, speech={r['has_speech']}, {len(r['reps'])} representative frames")
        for u in r['units'][:12]:
            print(f"  [{V.fmt(u['start'])}] ({u['kind']}) {u['text']}")

In [ ]:
if HAS_VIDEO:
    vlm = V.VLM()
    vrag = V.VideoRAG(recs, clip, emb, vlm)
    qa_v = load_json('Data/video_qa.json')
    demo_turns = qa_v['dialogues'][0]['turns'] if qa_v.get('dialogues') else [{'question': '這些影片在做什麼？'}, {'question': '第一個步驟是什麼？'}]
    hist = []
    for t in demo_turns:
        res = vrag.answer(t['question'], hist)
        hist.append((t['question'], res['answer']))
        print(f"使用者：{t['question']}\n  改寫：{res['standalone']}\n  候選影片：{res['video_scores']}\n  助理：{res['answer']}")
        print('  使用的畫面：', [f"{f['video']} {V.fmt(f['t'])}" for f in res['frames']])

---
## 作業三：評估 RAG 系統
### 基礎：以 RAGAS 評估作業一
指標依 RAGAS 的定義：**faithfulness**（回答拆成陳述，有幾成能由檢索內容推得）、**answer relevancy**（由回答反推問題，與原問題的相似度）、**context precision**（有用的 chunk 是否排在前面）、**context recall**（標準答案的陳述有幾成在檢索內容中）、**answer correctness**（回答與標準答案的陳述層級 F1）。為了不超過 Gemini 免費額度，每筆回答只呼叫一次評審，同時取得所有判斷；分數由程式依判斷計算。上面作業一已評估 47 題 × 5 個系統，這裡再評估 33 輪多輪對話，並檢查評審的可靠度（同一批回答由 Gemini 與本地 Qwen3-4B 各評一次）。

In [ ]:
# GPU: free the video models before Qwen3-4B (judge, answer relevancy) is loaded again
if HAS_VIDEO and 'vlm' in globals():
    vrag.vlm = None; del vlm; free_gpu()
dialog_recs = [dict(r, id=f"{did}-{r['turn']}") for did, v in dialog_res.items() for r in v]
ragas_dfs['multi-turn (grounded)'], _ = ragas_eval.evaluate('dialogues', dialog_recs, judge, llm, emb)
S = ragas_eval.summarize(list(ragas_dfs.values()))
display(S.round(3))
if judge.gemini is not None:
    D = ragas_eval.judge_agreement(answers['grounded'], judge, ragas_eval.Judge(None, llm), llm, emb, 'grounded')
    display(D.round(3))

In [ ]:
import matplotlib.pyplot as plt
M = ragas_eval.METRICS
fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(M))
for i, (_, r) in enumerate(S.iterrows()):
    ax.bar(x + (i - len(S) / 2) * 0.13, [r[m] for m in M], 0.13, label=r['system'])
ax.set_xticks(x); ax.set_xticklabels(M); ax.set_ylim(0, 1.05); ax.legend(fontsize=8, ncol=3); ax.set_title('RAGAS metrics of the HW08 part-1 RAG system')
plt.tight_layout(); plt.savefig(out_path('hw3_ragas.png'), dpi=130); plt.show()

### 進階：延伸到 VideoRAG 的評估
在 RAGAS 之外加上影片特有的面向：**影片檢索**（video hit@1/3）、**時間定位**（使用的片段 / 畫面是否落在標準答案的時間區間、引用的時間點是否正確、送進模型的畫面有幾成在區間內）、**多模態 RAGAS**（Gemini 直接看選出的畫面判斷回答是否有根據、畫面是否有用）。並做兩組消融：模態（畫面 + 文字 / 只有文字 / 只有畫面）與畫面挑選（VideoRAG 挑選 vs 均勻取樣），依問題類型（語音 / 畫面）分開統計。

In [ ]:
from Code import video_eval
if HAS_VIDEO:
    llm.unload(); free_gpu()                       # GPU: Qwen3-4B out, Qwen3-VL back in
    if getattr(vrag, 'vlm', None) is None:
        vlm = V.VLM(); vrag.vlm = vlm
    VS, VT = video_eval.evaluate(vrag, gemini if gemini.available or gemini.cache else None)
    if VS is not None:
        display(VS.round(3)); display(VT.round(3))

---
## 作業四：GraphRAG
論文：**Self-RAG**（Asai et al., 2023，arXiv 2310.11511，CC BY 4.0）。Qwen3-4B 從每個 1,200 字的段落抽出實體與關係 → 名稱正規化合併 → networkx 圖 → Louvain 社群 → 每個社群一份報告。查詢：**local**（相似實體 + 一階鄰居關係 + 來源段落）、**global**（每個社群報告先回答並評分，再整合）、**graph**（由 LLM 判斷題型後選 local 或 global）；對照組 **vector**（一般 RAG：前 5 個段落）。Benchmark：25 題（local 14、multi-hop 7、global 4），每題列出關鍵要點，評審逐點檢查。

In [ ]:
from Code import graphrag as GR
from Code.graph_data import GRAPH_QA
if HAS_VIDEO and 'vlm' in globals():
    vrag.vlm = None; del vlm; free_gpu()           # GPU: free Qwen3-VL before Qwen3-4B builds the graph
gindex = GR.GraphIndex(llm, emb).build()
display(pd.Series(gindex.stats(), name='value'))
for c in gindex.comms[:6]:
    print(f"[community {c['id']}] {c['title']} ({len(c['members'])} entities)\n   {c['summary'][:300]}")
gindex.plot()

In [ ]:
qs = [g['question'] for g in GRAPH_QA]
graph_recs = {}
for mode in ('vector', 'local', 'global', 'graph'):
    p = out_path(f'hw4_answers_{mode}.json')
    graph_recs[mode] = load_json(p) if os.path.exists(p) else gindex.answer(qs, mode)
    save_json(graph_recs[mode], p)
judge_fn = (lambda k, p: gemini.judge_obj(k, p)) if (gemini.available or gemini.cache) else \
           (lambda k, p: GR.parse_json(llm.one([{'role': 'user', 'content': p}], 300), None))
GS, GT, Gdf = GR.evaluate(graph_recs, GRAPH_QA, judge_fn)
display(GS.round(3)); display(GT.round(3))
print('routing of the graph mode:', pd.Series([r['route'] for r in graph_recs['graph']]).value_counts().to_dict())
for g, a, b in list(zip(GRAPH_QA, graph_recs['vector'], graph_recs['graph']))[21:23]:
    print('=' * 100); print(g['question']); print('  vector:', a['answer'][:300]); print('  graph :', b['answer'][:300])

### 打包結果
`Output/`（所有表格、圖、回答、評審快取）壓縮成 `HW08_results.zip`（不含 embedding 快取與影片索引的圖片）。

In [ ]:
import zipfile, glob
with zipfile.ZipFile('HW08_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('Output/**', recursive=True):
        if os.path.isfile(f) and '/emb_cache/' not in f.replace(os.sep, '/') and not f.endswith(('.npy', '.jpg')):
            z.write(f)
print(f"HW08_results.zip: {os.path.getsize('HW08_results.zip') / 2**20:.1f} MiB")
if IN_COLAB:
    from google.colab import files
    files.download('HW08_results.zip')

### 作業一進階 + 作業二：使用者介面
一個介面兩個分頁。**勞動法規 RAG**：左側對話；右側顯示系統實際用來檢索的問題（多輪改寫後）、檢索到的條文（★ = 回答中有引用，條號連到全國法規資料庫）與相似度，並可即時調整 Top-K、Prompting technique、Query 轉換、Re-ranker。**影片 VideoRAG**：右側顯示 Frame Selection 選出的畫面（影片名稱與時間）與使用的逐字稿 / 畫面描述。

T4 放不下 Qwen3-4B 與 Qwen3-VL 兩個模型，所以每個分頁回答前會先卸載另一個分頁的模型（切換分頁後的第一個問題要多等約一分鐘載入）。

In [ ]:
from Code.ui import build_demo
video_rag = None
if HAS_VIDEO:
    if vrag.vlm is None:
        vrag.vlm = V.VLM(load=False)          # loaded on the first video question
    video_rag = vrag
build_demo(rag, video_rag).launch(share=IN_COLAB, debug=False)